# 03. Cleaning and Standardising Tabular Data
**Engineering Data Processing with Python** | one-day course | approx. 60 minutes

> **SOLUTION NOTEBOOK.** Every blank and TODO is filled in. Try the exercise version first; use this when you are stuck or to compare approaches. There is usually more than one correct answer.


In [1]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## What you will do

Take the Kilbrack work orders and asset register from "loads without error" to "safe to analyse":

1. Tidy text: spaces, case, column names.
2. Map messy categories to a small agreed set, and catch anything unmapped.
3. Turn numbers stored as text into real numbers, and prove nothing was lost.
4. Parse dates in more than one format.
5. Normalise units (W and kW) into one column.
6. Decide what to do about missing values, on purpose.
7. Save clean outputs.

We do **not** remove duplicates here. That needs its own care (Module 04).

## 0. Load the raw data

Keep an untouched copy (`wo_raw`) so you can always compare before and after.

In [2]:
wo_raw = pd.read_csv("data/work_orders_cmms.csv", na_values=["-"])
wo = wo_raw.copy()
print(wo.shape)
wo.head()

(1016, 12)


,wo_id,asset_id,wo_type,priority,date_raised,date_closed,status,labour_hours,parts_cost_eur,technician,description,last_modified
0,WO-25-00270,PMP-005,PM,3,23/06/2025 13:15,24/06/2025 01:17,Closed,2.5,222.81,Gráinne FitzGerald,Quarterly PM service,2025-06-24 03:41:14
1,WO-26-00692,PMP-005,pm,4,04/03/2026 18:45,06/03/2026 06:12,CLOSED,1.5,0.77,Orla Doyle,"Replace filters, check belts",2026-03-06 09:02:03
2,WO-25-00449,PMP-001,Inspection,3,10/10/2025 10:30,11/10/2025 08:37,Closed,1.5,€0.00,Ciarán Walsh,Vibration route survey,2025-10-11 09:35:23
3,WO-25-00466,PMP-009,PM,4,18/10/2025 07:00,18/10/2025 13:04,closed,1.0,€7.48,Patrick Ryan,Annual service per OEM schedule,2025-10-18 17:15:07
4,WO-25-00198,CNV-006,Corrective,4,06/05/2025 17:45,08/05/2025 19:56,Closed,1.5,€417.25,Niamh O Connor,"Belt snapped, replaced",2025-05-08 20:18:16


## 1. Tidy text

pandas text methods live under `.str`. They work on the whole column at once.

### Exercise 3.1 [Level 1: fill the blanks]
Asset IDs: remove surrounding spaces and make upper case. Status: remove spaces and use Title Case.

In [3]:
print("Distinct asset IDs before:", wo["asset_id"].nunique())
wo["asset_id"] = wo["asset_id"].str.strip().str.upper()
print("Distinct asset IDs after: ", wo["asset_id"].nunique())

print(wo["status"].value_counts(), "\n")
wo["status"] = wo["status"].str.strip().str.title()
print(wo["status"].value_counts())

Distinct asset IDs before: 159
Distinct asset IDs after:  70
status
Closed         570
closed         216
CLOSED         194
Open            30
In Progress      6
Name: count, dtype: int64 

status
Closed         980
Open            30
In Progress      6
Name: count, dtype: int64


In [4]:
check(wo["asset_id"].str.match(r"^[A-Z]{3}-[0-9A-Z]{3}$").all() and set(wo["status"]) == {"Closed", "Open", "In Progress"},
      "IDs and status tidy. Distinct asset IDs dropped because 'cnc-001' and 'CNC-001 ' are now the same asset.",
      "Check strip/upper on asset_id and strip/title on status.")

PASS: IDs and status tidy. Distinct asset IDs dropped because 'cnc-001' and 'CNC-001 ' are now the same asset.


## 2. Map messy categories

There are 12 spellings of 3 work order types. Fix it in two steps: normalise (strip + upper) so similar spellings
collapse, then **map** each remaining spelling to an agreed code using a dictionary.

`.map()` has a useful property: anything **not** in the dictionary becomes `NaN`. That is a feature. When next
month's export contains a new spelling, you find out immediately instead of it slipping through.

### Exercise 3.2 [Level 1: fill the blanks]

In [5]:
wo["wo_type"].str.strip().str.upper().value_counts()

wo_type
PM            505
CM            189
CORRECTIVE     98
PREVENTIVE     91
INSP           71
BREAKDOWN      42
INSPECTION     20
Name: count, dtype: int64

In [6]:
TYPE_MAP = {
    "PM": "PM",
    "PREVENTIVE": "PM",
    "CM": "CM",
    "CORRECTIVE": "CM",
    "BREAKDOWN": "CM",
    "INSP": "INSP",
    "INSPECTION": "INSP",
}
wo["wo_type"] = wo["wo_type"].str.strip().str.upper().map(TYPE_MAP)

unmapped = wo["wo_type"].isna().sum()
print("Unmapped types:", unmapped)
wo["wo_type"].value_counts()

Unmapped types: 0


wo_type
PM      596
CM      329
INSP     91
Name: count, dtype: int64

In [7]:
check(unmapped == 0 and set(wo["wo_type"]) == {"PM", "CM", "INSP"}, "Three clean categories.",
      "Something did not map. Look at the value_counts above the dictionary.")

PASS: Three clean categories.


## 3. Numbers stored as text

### Exercise 3.3 [Level 1: fill the blanks]
Parts cost arrives as `222.81`, `€1,250.00` and missing markers. Remove the `€` and the thousands comma, then
convert. Then **prove** that conversion did not create any new missing values.

In [8]:
missing_before = wo["parts_cost_eur"].isna().sum()

cost_text = (wo["parts_cost_eur"]
             .str.replace("€", "", regex=False)
             .str.replace(",", "", regex=False)
             .str.strip())
wo["parts_cost_eur"] = pd.to_numeric(cost_text, errors="coerce")

missing_after = wo["parts_cost_eur"].isna().sum()
print(f"Missing before: {missing_before}, after: {missing_after}")
wo["parts_cost_eur"].describe()

Missing before: 92, after: 92


count      924.000000
mean       362.730736
std        872.319871
min       -150.000000
25%         63.745000
50%        173.305000
75%        277.805000
max      11678.260000
Name: parts_cost_eur, dtype: float64

In [9]:
check(missing_before == missing_after and wo["parts_cost_eur"].dtype == "float64",
      "Converted with no hidden losses. (If 'after' were bigger, some text failed to convert and became NaN.)",
      "Missing count changed or column not numeric. Look at which values failed.")

PASS: Converted with no hidden losses. (If 'after' were bigger, some text failed to convert and became NaN.)


That before/after comparison is the most useful habit in this course. `errors="coerce"` is powerful because it never
crashes, and dangerous for the same reason.

Notice the minimum cost is negative (`-150`). That is probably a credit note. We leave it and flag it in Module 06.

### Exercise 3.4 [Level 2: finish the code]
Priority should be 1 to 4. The export has `1`, `P1`, `Urgent` and blanks.

* Make it text, strip, upper case.
* Remove the letter `P`.
* Replace `URGENT` with `1` (Kilbrack's procedure says Urgent = P1).
* Convert to numbers, then to pandas' nullable integer type `"Int64"` (capital I), which allows missing values.

In [10]:
wo["priority"].astype(str).str.strip().str.upper().value_counts()

priority
3      452
4      397
2       84
1       35
NAN     21
P4      14
P3      11
P2       2
Name: count, dtype: int64

In [11]:
p = wo["priority"].astype("string").str.strip().str.upper()
p = p.str.replace("P", "", regex=False).str.replace("URGENT", "1", regex=False)
wo["priority"] = pd.to_numeric(p, errors="coerce").astype("Int64")

wo["priority"].value_counts(dropna=False)

priority
3       463
4       411
2        86
1        35
<NA>     21
Name: count, dtype: Int64

In [12]:
vals = set(wo["priority"].dropna().unique())
check(vals == {1, 2, 3, 4} and str(wo["priority"].dtype) == "Int64",
      "Priority is 1 to 4, blanks stay missing.", "Expected only 1, 2, 3, 4 (plus missing) and dtype Int64.")

PASS: Priority is 1 to 4, blanks stay missing.


In [13]:
wo["labour_hours"] = pd.to_numeric(wo["labour_hours"], errors="coerce")
wo["labour_hours"].describe()

count    980.000000
mean       4.402551
std       14.907698
min       -4.000000
25%        2.000000
50%        3.500000
75%        4.500000
max      360.000000
Name: labour_hours, dtype: float64

Labour hours run from negative to 360. A single work order of 360 hours is unlikely; 360 *minutes* typed into an
hours field is very likely. We do not fix that by guessing here. We flag it in Module 06.

## 4. Dates in more than one format

In Module 02 you parsed with one format, then a second. Turn that into a reusable function.

### Exercise 3.5 [Level 2: finish the code]
`parse_dates(series, formats)` should try each format in turn, only filling values that are still missing.

In [14]:
def parse_dates(series, formats):
    """Parse text dates trying each format in order. Returns a datetime Series; unparseable values stay NaT."""
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in formats:
        parsed = pd.to_datetime(series, format=fmt, errors="coerce")
        result = result.fillna(parsed)
    return result

DATE_FORMATS = ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"]
blank_closed = wo_raw["date_closed"].isna().sum()

wo["date_raised"] = parse_dates(wo["date_raised"], DATE_FORMATS)
wo["date_closed"] = parse_dates(wo["date_closed"], DATE_FORMATS)
wo["last_modified"] = parse_dates(wo["last_modified"], ["%Y-%m-%d %H:%M:%S"])
print("Unparsed raised:", wo["date_raised"].isna().sum())
print("Unparsed closed:", wo["date_closed"].isna().sum(), "(blank in the raw file:", blank_closed, ")")

Unparsed raised: 0
Unparsed closed: 41 (blank in the raw file: 41 )


In [15]:
check(wo["date_raised"].isna().sum() == 0 and wo["date_closed"].isna().sum() == blank_closed,
      "All dates parsed. The only missing closed dates are ones that were blank to begin with.",
      "Some dates did not parse. Is parse_dates filling result from each format?")

PASS: All dates parsed. The only missing closed dates are ones that were blank to begin with.


## 5. Missing values: decide, do not default

| Column | Missing means | Decision |
|---|---|---|
| `date_closed` | Work order still open (or someone forgot) | Keep missing. Module 06 flags *Closed* with no date |
| `parts_cost_eur` | Not recorded | Keep missing. Filling with the average would **invent money** |
| `labour_hours` | Not recorded | Keep missing, add a flag. An average would bias repair time KPIs |
| `priority` | Not recorded | Keep missing |

The common mistake is `df.fillna(0)` or filling with the mean so the numbers "work". That makes missing data look
like real data, and nobody downstream can tell the difference.

In [16]:
wo["hours_missing"] = wo["labour_hours"].isna()
wo.isna().sum()

wo_id              0
asset_id           0
wo_type            0
priority          21
date_raised        0
date_closed       41
status             0
labour_hours      36
parts_cost_eur    92
technician         0
description        0
last_modified      0
hours_missing      0
dtype: int64

## 6. The asset register

### Exercise 3.6 [Level 1: fill the blanks]
Load it (remember the title rows) and convert the column names to `snake_case`: lower case, no spaces.

In [17]:
reg_raw = pd.read_excel("data/asset_register.xlsx", skiprows=3)
reg = reg_raw.copy()

reg.columns = (reg.columns.str.strip()
                          .str.lower()
                          .str.replace(r"[^a-z0-9]+", "_", regex=True))
reg["asset_id"] = reg["asset_id"].str.strip().str.upper()
reg.columns.tolist()

['asset_id',
 'asset_description',
 'asset_type',
 'site',
 'install_date',
 'manufacturer',
 'rated_power',
 'criticality',
 'status']

In [18]:
check(reg.columns.tolist() == ["asset_id", "asset_description", "asset_type", "site", "install_date",
                               "manufacturer", "rated_power", "criticality", "status"],
      "Consistent column names.", "Expected names like asset_id, install_date, rated_power.")

PASS: Consistent column names.


`r"[^a-z0-9]+"` is a *regular expression* meaning "one or more characters that are not a lower-case letter or digit".
Regular expressions are a good thing to ask Gemini to explain piece by piece.

### Exercise 3.7 [Level 2: finish the code]
Sites come as `Cork`, `cork`, `CORK `, `Cork Plant`, `Limerick Site` and so on. Normalise case and spaces first,
then map the leftovers. Finish `SITE_MAP` so every row ends up as `Cork`, `Galway` or `Limerick`.

In [19]:
reg["site"].str.strip().str.title().value_counts()

site
Cork             20
Galway           17
Limerick         17
Cork Plant        6
Limerick Site     2
Name: count, dtype: int64

In [20]:
SITE_MAP = {
    "Cork": "Cork",
    "Cork Plant": "Cork",
    "Galway": "Galway",
    "Limerick": "Limerick",
    "Limerick Site": "Limerick",
}
reg["site"] = reg["site"].str.strip().str.title().map(SITE_MAP)
print("Unmapped sites:", reg["site"].isna().sum())
reg["site"].value_counts()

Unmapped sites: 0


site
Cork        26
Limerick    19
Galway      17
Name: count, dtype: int64

In [21]:
check(set(reg["site"]) == {"Cork", "Galway", "Limerick"}, "Three sites.", "Some sites are unmapped (NaN).")

PASS: Three sites.


### Exercise 3.8 [Level 2: finish the code]  Units: rated power to kW

Values look like `38 kW`, `38kW`, `38 KW`, `38000 W`, `38` (a number, assume kW) and blanks.
Write `to_kw(value)` that returns a float in kW, or `np.nan` if it is blank.

Approach: if it is already a number, return it. Otherwise make it upper case text without spaces, then check the
ending: `KW` (strip it off) or `W` (strip it off and divide by 1000). Order matters: why check `KW` before `W`?

In [22]:
def to_kw(value):
    """Convert a rated power value (number or text with W/kW) to kW."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).upper().replace(" ", "")
    if text.endswith("KW"):
        return float(text[:-2])
    elif text.endswith("W"):
        return float(text[:-1]) / 1000
    return float(text)

reg["rated_power_kw"] = reg["rated_power"].apply(to_kw)
reg[["asset_id", "rated_power", "rated_power_kw"]].head(12)

,asset_id,rated_power,rated_power_kw
0,CNC-001,38 kW,38.0
1,CNC-002,36kW,36.0
2,CNC-003,38000 W,38.0
3,CNC-004,28.5,28.5
4,CNC-005,34.5 kW,34.5
5,CNC-006,22,22.0
6,CNC-007,40 KW,40.0
7,CNC-008,25.5 kW,25.5
8,CNC-009,38.5kW,38.5
9,CNC-010,16500 W,16.5


In [23]:
kw = reg["rated_power_kw"]
check(kw.notna().sum() == reg["rated_power"].notna().sum() and kw.notna().any() and kw.dropna().between(1, 250).all(),
      "Every power value converted, all between 1 and 250 kW. An engineering sense check as well as a code check.",
      "Some values did not convert, or a W value was not divided by 1000.")

PASS: Every power value converted, all between 1 and 250 kW. An engineering sense check as well as a code check.


*Why `KW` before `W`?* Because `"38KW"` also ends with `"W"`. Check the longer suffix first or you would try
`float("38K")` and fail.

### Exercise 3.9 [Level 3: write it yourself, Gemini allowed]  Install dates

`install_date` contains real Excel dates plus text like `22/06/2019`, `2016-01-01`, `08.06.2014` and `March 2016`.
One asset has an install date in 2031.

Write code that:
1. Creates `install_date_clean` as proper datetimes (all formats handled, nothing lost along the way).
2. Creates `install_date_suspect` = True where the date is in the future.

**Prompt to try:**
> I have a pandas column install_date with a mix of datetime objects and text dates in these formats:
> dd/mm/yyyy, yyyy-mm-dd, dd.mm.yyyy and 'Month yyyy'. Write code to convert it to datetime without guessing
> day/month order, and tell me how to check that no values were lost.

**Check it:** count non-missing before and after. Look at `reg[["install_date", "install_date_clean"]]` for the
odd ones. Did Gemini use `dayfirst` or `format="mixed"`? Is that safe here?

In [24]:
def parse_install(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (pd.Timestamp, np.datetime64)) or hasattr(value, "year"):
        return pd.Timestamp(value)
    for fmt in ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%B %Y"]:
        try:
            return pd.to_datetime(str(value).strip(), format=fmt)
        except ValueError:
            continue
    return pd.NaT

reg["install_date_clean"] = reg["install_date"].apply(parse_install)
reg["install_date_suspect"] = reg["install_date_clean"] > pd.Timestamp.today()
print("Non-missing before:", reg["install_date"].notna().sum(), " after:", reg["install_date_clean"].notna().sum())
reg.loc[reg["install_date_suspect"], ["asset_id", "install_date", "install_date_clean"]]

Non-missing before: 62  after: 62


,asset_id,install_date,install_date_clean
19,CMP-006,2031-04-12 00:00:00,2031-04-12


In [25]:
check("install_date_clean" in reg.columns and reg["install_date_clean"].notna().sum() == reg["install_date"].notna().sum()
      and reg.get("install_date_suspect", pd.Series(dtype=bool)).sum() == 1,
      "All install dates parsed and one future date flagged.",
      "Need install_date_clean with nothing lost, and install_date_suspect flagging exactly one asset.")

PASS: All install dates parsed and one future date flagged.


### Exercise 3.10 [Level 1: fill the blanks]  Criticality
The standard is A, B, C. Some rows say `High` or `Low`, two are blank. Map the words, and make blanks explicit.

In [26]:
reg["criticality"] = reg["criticality"].replace({"High": "A", "Medium": "B", "Low": "C"})
reg["criticality"] = reg["criticality"].fillna("Unassigned")
reg["criticality"].value_counts()

criticality
B             31
C             17
A             12
Unassigned     2
Name: count, dtype: int64

Filling with `"Unassigned"` is different from filling a number with an average: it makes the gap **more** visible,
not less. Anyone who filters for A, B or C will not accidentally include these.

## 7. Save the clean outputs

We keep the original columns we still need and drop the raw text versions we replaced.

In [27]:
wo_clean = wo[["wo_id", "asset_id", "wo_type", "priority", "date_raised", "date_closed", "status",
               "labour_hours", "hours_missing", "parts_cost_eur", "technician", "description", "last_modified"]]

reg_clean = reg.drop(columns=["rated_power", "install_date"]).rename(columns={"install_date_clean": "install_date"})

wo_clean.to_csv("outputs/work_orders_clean.csv", index=False)
reg_clean.to_csv("outputs/asset_register_clean.csv", index=False)
print(wo_clean.shape, reg_clean.shape)
reg_clean.head()

(1016, 13) (62, 10)


,asset_id,asset_description,asset_type,site,manufacturer,criticality,status,rated_power_kw,install_date,install_date_suspect
0,CNC-001,CNC Mill 1,CNC Mill,Galway,Hartwell Machine Tools,C,In Service,38.0,2018-06-19,False
1,CNC-002,CNC Mill 2,CNC Mill,Cork,Hartwell Machine Tools,C,In Service,36.0,2011-03-22,False
2,CNC-003,CNC Mill 3,CNC Mill,Limerick,Hartwell Machine Tools,A,In Service,38.0,2019-06-22,False
3,CNC-004,CNC Mill 4,CNC Mill,Galway,Hartwell Machine Tools,C,In Service,28.5,2016-01-01,False
4,CNC-005,CNC Mill 5,CNC Mill,Limerick,Hartwell Machine Tools,B,In Service,34.5,2014-06-08,False


Remember: this Colab session is temporary. If you want to keep these files, download them from the Files panel.
The next notebook starts from a trainer checkpoint copy of the same outputs so everyone is on the same page.

## Summary

* `.str.strip().str.upper()` / `.str.title()` fix most text noise.
* Map categories with a dictionary, and **count what did not map**.
* After converting text to numbers, compare missing counts before and after.
* Parse dates with explicit formats in a loop; anything left over is something to investigate.
* Units: convert to one unit in a new column with the unit in its name.
* Missing values are a decision. Do not invent data to make numbers work.

Next: **04 Deduplication and Record Linkage**.